In [1]:
from __future__ import annotations

import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag


BASE_URL = "https://www.transfermarkt.com"


TRANSFERMARKT_LEAGUES = {
    "netherlands_eredivisie": {
        "league_name": "Eredivisie",
        "slug": "eredivisie",
        "competition_code": "NL1",
    },
    "england_premier_league": {
        "league_name": "Premier League",
        "slug": "premier-league",
        "competition_code": "GB1",
    },
    "england_championship": {
        "league_name": "Championship",
        "slug": "championship",
        "competition_code": "GB2",
    },
}


def build_transfermarkt_transfers_url(
    slug: str,
    competition_code: str,
    season_start_year: int,
    winter_only: bool = False,
) -> str:
    if winter_only:
        return (
            f"{BASE_URL}/{slug}/transfers/wettbewerb/"
            f"{competition_code}/saison_id/{season_start_year}/s_w/w"
        )

    return (
        f"{BASE_URL}/{slug}/transfers/wettbewerb/"
        f"{competition_code}/saison_id/{season_start_year}"
    )


def fetch_html(url: str) -> str:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/125.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    }

    time.sleep(2)
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.text


def clean_text(value: str | None) -> str | None:
    if value is None:
        return None

    value = re.sub(r"\s+", " ", value.replace("\xa0", " ")).strip()
    return value or None


def extract_id_from_href(href: str | None, pattern: str) -> str | None:
    if not href:
        return None

    match = re.search(pattern, href)
    return match.group(1) if match else None


def get_player_link(row: Tag) -> Tag | None:
    link = row.select_one("td.spieler-transfer-cell span.hide-for-small a")
    if link:
        return link

    first_td = row.find("td")
    return first_td.find("a") if first_td else None


def parse_nationalities(td: Tag) -> list[str]:
    nationalities = []

    for img in td.find_all("img"):
        value = clean_text(img.get("title") or img.get("alt"))
        if value:
            nationalities.append(value)

    return nationalities


def parse_other_club(tds: list[Tag]) -> tuple[str | None, str | None, str | None]:
    club_td = tds[7] if len(tds) > 7 else None
    club_link = club_td.find("a") if club_td else None

    club_name = None
    if club_link:
        club_name = clean_text(club_link.get("title") or club_link.get_text(" ", strip=True))

    club_href = club_link.get("href") if club_link else None
    club_id = extract_id_from_href(club_href, r"/verein/(\d+)")

    country = None
    if club_td:
        flag = club_td.find("img")
        country = clean_text(flag.get("title") or flag.get("alt")) if flag else None

    return club_name, club_id, country


def parse_transfer_table(
    table: Tag,
    direction: str,
    league_name: str,
    competition_code: str,
    season_start_year: int,
    club_name: str,
    club_id: str | None,
) -> list[dict]:
    records = []

    for row in table.select("tbody tr"):
        tds = row.find_all("td", recursive=False)

        if len(tds) < 9:
            continue

        player_link = get_player_link(row)
        player_href = player_link.get("href") if player_link else None

        player_name = None
        if player_link:
            player_name = clean_text(
                player_link.get("title") or player_link.get_text(" ", strip=True)
            )

        other_club_name, other_club_id, other_club_country = parse_other_club(tds)

        fee_link = tds[8].find("a")
        fee_href = fee_link.get("href") if fee_link else None

        if direction == "in":
            from_club_name = other_club_name
            from_club_id = other_club_id
            from_club_country = other_club_country
            to_club_name = club_name
            to_club_id = club_id
        else:
            from_club_name = club_name
            from_club_id = club_id
            from_club_country = None
            to_club_name = other_club_name
            to_club_id = other_club_id

        records.append(
            {
                "league_name": league_name,
                "competition_code": competition_code,
                "season_start_year": season_start_year,
                "season_name": f"{str(season_start_year)[-2:]}/{str(season_start_year + 1)[-2:]}",
                "club_name": club_name,
                "club_id": club_id,
                "direction": direction,
                "player_name": player_name,
                "player_id": extract_id_from_href(player_href, r"/spieler/(\d+)"),
                "player_url": urljoin(BASE_URL, player_href) if player_href else None,
                "age": clean_text(tds[1].get_text(" ", strip=True)),
                "nationalities": parse_nationalities(tds[2]),
                "position": clean_text(tds[3].get_text(" ", strip=True)),
                "short_position": clean_text(tds[4].get_text(" ", strip=True)),
                "market_value": clean_text(tds[5].get_text(" ", strip=True)),
                "from_club_name": from_club_name,
                "from_club_id": from_club_id,
                "from_club_country": from_club_country,
                "to_club_name": to_club_name,
                "to_club_id": to_club_id,
                "to_club_country": other_club_country if direction == "out" else None,
                "fee": clean_text(tds[8].get_text(" ", strip=True)),
                "transfer_id": extract_id_from_href(fee_href, r"/transfer_id/(\d+)"),
                "transfer_url": urljoin(BASE_URL, fee_href) if fee_href else None,
            }
        )

    return records


def parse_league_transfers(
    html: str,
    league_name: str,
    competition_code: str,
    season_start_year: int,
) -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    records = []

    for box in soup.select("div.box"):
        club_header = box.select_one("h2.content-box-headline--logo")

        if not club_header:
            continue

        club_link = club_header.find("a", href=re.compile(r"/transfers/verein/"))

        club_name = None
        if club_link:
            club_name = clean_text(
                club_link.get("title") or club_link.get_text(" ", strip=True)
            )

        if club_name:
            club_name = club_name.replace("Array", "").strip()

        club_href = club_link.get("href") if club_link else None
        club_id = extract_id_from_href(club_href, r"/verein/(\d+)")

        for table in box.select("div.responsive-table table"):
            first_header = table.select_one("thead th")

            if not first_header:
                continue

            first_header_text = clean_text(first_header.get_text(" ", strip=True))

            if first_header_text == "In":
                direction = "in"
            elif first_header_text == "Out":
                direction = "out"
            else:
                continue

            records.extend(
                parse_transfer_table(
                    table=table,
                    direction=direction,
                    league_name=league_name,
                    competition_code=competition_code,
                    season_start_year=season_start_year,
                    club_name=club_name,
                    club_id=club_id,
                )
            )

    return pd.DataFrame(records)


def scrape_league_transfers(
    league_key: str,
    season_start_year: int,
    winter_only: bool = False,
) -> pd.DataFrame:
    league = TRANSFERMARKT_LEAGUES[league_key]

    url = build_transfermarkt_transfers_url(
        slug=league["slug"],
        competition_code=league["competition_code"],
        season_start_year=season_start_year,
        winter_only=winter_only,
    )

    html = fetch_html(url)

    return parse_league_transfers(
        html=html,
        league_name=league["league_name"],
        competition_code=league["competition_code"],
        season_start_year=season_start_year,
    )


df = scrape_league_transfers(
    league_key="netherlands_eredivisie",
    season_start_year=2025,
)

# df.to_csv("eredivisie_transfers_2025_26.csv", index=False)
print(df.head())

  league_name competition_code  season_start_year season_name       club_name  \
0  Eredivisie              NL1               2025       25/26  Ajax Amsterdam   
1  Eredivisie              NL1               2025       25/26  Ajax Amsterdam   
2  Eredivisie              NL1               2025       25/26  Ajax Amsterdam   
3  Eredivisie              NL1               2025       25/26  Ajax Amsterdam   
4  Eredivisie              NL1               2025       25/26  Ajax Amsterdam   

  club_id direction     player_name player_id  \
0     610        in    Oscar Gloukh    930571   
1     610        in       Raúl Moro    624942   
2     610        in      Ko Itakura    355816   
3     610        in  Kasper Dolberg    283196   
4     610        in   Maher Carrizo   1162160   

                                          player_url  ... market_value  \
0  https://www.transfermarkt.com/oscar-gloukh/pro...  ...      €20.00m   
1  https://www.transfermarkt.com/raul-moro/profil...  ...       €7.50m

In [4]:
import re
import numpy as np
import pandas as pd


def parse_transfermarkt_fee_to_eur(fee: str | None) -> float | None:
    """
    Converts Transfermarkt fee strings into EUR numeric values.

    Examples:
    '€2.50m' -> 2500000
    '€500k' -> 500000
    'Loan fee: €250k' -> 250000
    'free transfer' -> 0
    'loan transfer' -> 0
    '-' -> 0
    '?' -> None
    """
    if fee is None or pd.isna(fee):
        return 0

    fee_text = str(fee).strip().lower()

    if fee_text in {"", "-", "free transfer", "loan transfer", "end of loan", "end of loanJun 30, 2025".lower()}:
        return 0

    if "?" in fee_text:
        return None

    # Find first euro amount inside the string
    # Works for: €2.50m, €500k, Loan fee: €250k
    match = re.search(r"€\s?([\d,.]+)\s?([mk])?", fee_text)

    if not match:
        return 0

    number = match.group(1).replace(",", ".")
    suffix = match.group(2)

    try:
        value = float(number)
    except ValueError:
        return None

    if suffix == "m":
        return value * 1_000_000

    if suffix == "k":
        return value * 1_000

    return value

In [5]:
transfers_df = df.copy()

transfers_df["fee_eur"] = transfers_df["fee"].apply(parse_transfermarkt_fee_to_eur)

transfers_df["expenditure_eur"] = np.where(
    transfers_df["direction"].eq("in"),
    transfers_df["fee_eur"].fillna(0),
    0,
)

transfers_df["income_eur"] = np.where(
    transfers_df["direction"].eq("out"),
    transfers_df["fee_eur"].fillna(0),
    0,
)

club_transfer_balance_df = (
    transfers_df
    .groupby(
        [
            "league_name",
            "competition_code",
            "season_name",
            "season_start_year",
            "club_id",
            "club_name",
        ],
        dropna=False,
    )
    .agg(
        arrivals=("direction", lambda s: (s == "in").sum()),
        departures=("direction", lambda s: (s == "out").sum()),
        expenditure_eur=("expenditure_eur", "sum"),
        income_eur=("income_eur", "sum"),
    )
    .reset_index()
)

club_transfer_balance_df["balance_eur"] = (
    club_transfer_balance_df["income_eur"]
    - club_transfer_balance_df["expenditure_eur"]
)

club_transfer_balance_df = club_transfer_balance_df.sort_values(
    ["league_name", "season_start_year", "balance_eur"],
    ascending=[True, False, False],
)

In [6]:
club_transfer_balance_df

,league_name,competition_code,season_name,season_start_year,club_id,club_name,arrivals,departures,expenditure_eur,income_eur,balance_eur
8,Eredivisie,NL1,25/26,2025,234,Feyenoord Rotterdam,36,36,57450000.0,103135000.0,45685000.0
15,Eredivisie,NL1,25/26,2025,610,Ajax Amsterdam,23,22,58750000.0,102030000.0,43280000.0
0,Eredivisie,NL1,25/26,2025,1090,AZ Alkmaar,14,14,26800000.0,51800000.0,25000000.0
13,Eredivisie,NL1,25/26,2025,467,NEC Nijmegen,18,19,4600000.0,26800000.0,22200000.0
11,Eredivisie,NL1,25/26,2025,383,PSV Eindhoven,11,14,63800000.0,78000000.0,14200000.0
14,Eredivisie,NL1,25/26,2025,468,Sparta Rotterdam,22,20,1100000.0,9700000.0,8600000.0
7,Eredivisie,NL1,25/26,2025,202,FC Groningen,10,13,1413000.0,7450000.0,6037000.0
10,Eredivisie,NL1,25/26,2025,317,FC Twente Enschede,19,16,12350000.0,17600000.0,5250000.0
2,Eredivisie,NL1,25/26,2025,1304,Heracles Almelo,18,16,1500000.0,4500000.0,3000000.0
12,Eredivisie,NL1,25/26,2025,385,Fortuna Sittard,21,19,350000.0,2450000.0,2100000.0
